# PCNA MD Simulation — Pre-built system, run only

The system was already prepared locally (PDBFixer + solvation + CHARMM36m force field).  
This notebook only runs minimization → NVT → NPT → 100 ns production.

**Before running:** Go to `Runtime → Change runtime type → GPU (T4 or A100)`

**Files to upload (all in `data/md/` in the repo):**
- `1W60_solvated.pdb` (~25 MB)
- `system.xml` (~50 MB)
- `1W60_fixed.pdb` (~1 MB — used as topology reference)

**Total runtime: ~7–10 hours on T4. Use Colab Pro+ to avoid timeout.**

In [ ]:
# Cell 1: Install OpenMM (fast — pre-built wheel)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openmm', 'MDAnalysis'], check=True)

import openmm
from openmm import Platform
platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print('OpenMM', openmm.__version__, '| Platforms:', platforms)
assert 'CUDA' in platforms, 'No CUDA — switch to GPU runtime: Runtime → Change runtime type → GPU'

In [ ]:
# Cell 2: Upload pre-built system files
from google.colab import files
import os

print('Upload these 3 files from data/md/ in the GNN_PCNA repo:')
print('  1W60_solvated.pdb  (~25 MB)')
print('  system.xml         (~50 MB)')
print('  1W60_fixed.pdb     (~1 MB)')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))
assert 'system.xml' in uploaded, 'Missing system.xml'
assert '1W60_solvated.pdb' in uploaded, 'Missing 1W60_solvated.pdb'

In [ ]:
# Cell 3: Load pre-built system and set up simulation
from openmm.app import PDBFile, Simulation
from openmm import XmlSerializer, LangevinMiddleIntegrator, MonteCarloBarostat, Platform, unit

print('Loading topology...')
pdb = PDBFile('1W60_solvated.pdb')
print(f'  {pdb.topology.getNumAtoms():,} atoms, {pdb.topology.getNumResidues():,} residues')

print('Loading serialized system...')
with open('system.xml') as f:
    system = XmlSerializer.deserialize(f.read())
print('  System loaded')

integrator = LangevinMiddleIntegrator(
    310 * unit.kelvin,
    1.0 / unit.picosecond,
    4.0 * unit.femtosecond,
)

platform = Platform.getPlatformByName('CUDA')
properties = {'CudaPrecision': 'mixed'}

simulation = Simulation(pdb.topology, system, integrator, platform, properties)
simulation.context.setPositions(pdb.positions)
print('Simulation context created on CUDA')

In [ ]:
# Cell 4: Energy minimization + NVT equilibration (1 ns)
from openmm.app import StateDataReporter
import sys, time

print('Energy minimizing...')
simulation.minimizeEnergy(maxIterations=2000)
e = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f'  PE after minimization: {e}')

# Heat 100 K → 310 K in 500 ps
print('Heating 100 K → 310 K...')
simulation.context.setVelocitiesToTemperature(100)
for T in range(100, 311, 10):
    integrator.setTemperature(T * unit.kelvin)
    simulation.step(5_000)   # 20 ps per step

print('NVT at 310 K (remaining 500 ps)...')
simulation.reporters.append(
    StateDataReporter(sys.stdout, 50_000, step=True, temperature=True, speed=True))
simulation.step(125_000)   # 500 ps
print('NVT done')
simulation.saveCheckpoint('equil_nvt.chk')

In [ ]:
# Cell 5: NPT equilibration (1 ns, 310 K, 1 atm)
from openmm import MonteCarloBarostat

barostat = MonteCarloBarostat(1.0 * unit.bar, 310 * unit.kelvin, 25)
system.addForce(barostat)
simulation.context.reinitialize(preserveState=True)

print('NPT equilibration 1 ns...')
simulation.reporters.clear()
simulation.reporters.append(
    StateDataReporter(sys.stdout, 50_000,
        step=True, temperature=True, volume=True, density=True, speed=True))
simulation.step(250_000)   # 1 ns
print('NPT done')
simulation.saveCheckpoint('equil_npt.chk')

# Save topology with final equilibrated box for MDAnalysis
state = simulation.context.getState(getPositions=True)
with open('1W60_topology.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)
print('Saved 1W60_topology.pdb (MDAnalysis reference)')

In [ ]:
# Cell 6: 100 ns production run — DCD trajectory
from openmm.app import DCDReporter, CheckpointReporter
import time, sys

PROD_STEPS = 25_000_000   # 100 ns at 4 fs/step
SAVE_EVERY = 2_500        # 10 ps per frame  → 10,000 frames total

simulation.reporters.clear()
simulation.reporters.append(DCDReporter('1W60_production.dcd', SAVE_EVERY))
simulation.reporters.append(
    StateDataReporter('production.log', 250_000,
        step=True, time=True, potentialEnergy=True, temperature=True,
        volume=True, speed=True, progress=True,
        remainingTime=True, totalSteps=PROD_STEPS))
simulation.reporters.append(
    StateDataReporter(sys.stdout, 250_000,
        step=True, time=True, temperature=True, speed=True,
        progress=True, remainingTime=True, totalSteps=PROD_STEPS))
simulation.reporters.append(
    CheckpointReporter('production.chk', 2_500_000))  # checkpoint every 10 ns

print(f'Production: {PROD_STEPS:,} steps = 100 ns, saving every {SAVE_EVERY*4/1000:.0f} ps')
t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = time.time() - t0
print(f'Done in {elapsed/3600:.1f} h  |  {PROD_STEPS*4e-6/elapsed*3600:.1f} ns/day')

In [ ]:
# Cell 7: Quick RMSF check + save to Google Drive
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
import numpy as np, pandas as pd
from google.colab import drive
import os, shutil

# RMSF
u = mda.Universe('1W60_topology.pdb', '1W60_production.dcd')
print(f'Trajectory: {len(u.trajectory)} frames')
ref = mda.Universe('1W60_topology.pdb')
align.AlignTraj(u, ref, select='backbone', in_memory=False).run()
ca = u.select_atoms('name CA')
rmsf_vals = rms.RMSF(ca).run().rmsf

AOH = {25,26,27,38,39,40,41,42,44,45,46,47,123,125,126,128,231,232,233,234,250,251,252,253}
mask = np.array([r in AOH for r in ca.resids])
print(f'Pocket RMSF : {rmsf_vals[mask].mean():.3f} A')
print(f'Background  : {rmsf_vals[~mask].mean():.3f} A')
print(f'Fold-change : {rmsf_vals[mask].mean()/rmsf_vals[~mask].mean():.3f}')

pd.DataFrame({'resid': ca.resids, 'chain': ca.segids, 'rmsf_angstrom': rmsf_vals}).to_csv('1W60_rmsf.csv', index=False)

# Save to Google Drive
drive.mount('/content/drive')
outdir = '/content/drive/MyDrive/GNN_PCNA_MD'
os.makedirs(outdir, exist_ok=True)
for fn in ['1W60_production.dcd', '1W60_topology.pdb', '1W60_rmsf.csv', 'production.log', 'equil_npt.chk']:
    if os.path.exists(fn):
        shutil.copy(fn, outdir)
        print(f'Saved {fn} → Drive ({os.path.getsize(fn)/1e6:.0f} MB)')

## After Colab

1. Download from Google Drive: `GNN_PCNA_MD/1W60_production.dcd` and `1W60_topology.pdb`
2. Copy both to `GNN_PCNA/data/md/`
3. Run locally:
```bash
python scripts/run_md_analysis.py
```
This will compute RMSF, DCCM, and pocket volume and write results to `data/results/`.